In [1]:
import math, folium
import pandas as pd
import geopandas as gpd
import branca.colormap as cm
import folium.plugins as plugins

In [2]:
gdf_zips = gpd.read_file("/glade/work/cbecker/california_zip.geojson")

In [3]:
homes = pd.read_parquet("/glade/derecho/scratch/ggantos/CAT/home_locations_tao_nofilters/home_loc_by_row_group_comp.parquet")
homes_zips_count = homes.groupby("zipcode").size().sort_values(ascending=False)
print(homes_zips_count[:10])
print(homes_zips_count[-10:])

zipcode
90012    1116493
92335     272500
92101     229910
92503     218635
92704     205258
91766     204203
92336     200829
92703     199667
90044     177784
93702     171816
dtype: int64
zipcode
93651    791
93283    760
95311    714
93623    492
93207    319
93621    203
93652     94
93260     92
93255     89
95075     57
dtype: int64


In [ ]:
zip_code = "90011"
homes_zip = homes[homes.zipcode == zip_code].sort_values(by = ['MyCol', 'MyIdx'], ascending = [False, True])
homes_zip.shape
max_idx = 1000
vmax = math.ceil(homes_zip.dist_home.max() / 10.0) * 10
linear = cm.LinearColormap(["b", "r"],
                           caption="Distance between TAO and SKM home location",
                           vmin=1,
                           vmax=vmax,
                          )

m = folium.Map(location=[34.01, -118.26],
               zoom_start=14,
               control_scale=True,
               )

fg = folium.FeatureGroup(name="Home zipcode area").add_to(m)
# folium.Polygon(
#     locations=zip_locations,
#     fill_color="y",
#     fill_opacity=0.5,
#     color="y",
#     weight=1,
# ).add_to(fg)
folium.GeoJson(
    gdf_zips[gdf_zips.ZIP_CODE == zip_code],
    style_function=lambda feature: {
        "fillColor": "yellow",
        "fillOpacity": 0.7,
        "color": "yellow",
        "weight": 1,
    },
).add_to(fg)

fg = folium.FeatureGroup(name="Home Locations", show=True).add_to(m)
max_dist = homes_zip[:max_idx].dist_home.max()
for i, row in homes_zip[:max_idx].iterrows():
    folium.CircleMarker(
        location=row.loc[["lat", "lng"]].to_list(),
        fill_color=linear(row.dist_home),
        fill_opacity=0.5,
        color=linear(row.dist_home),
        radius=(row.dist_home/max_dist)*10,
        weight=1,
        highlight_function=lambda feature: {
            "color": "#78e05b",
            "weight": 2,
        },
        tooltip=row[["dist_home", "dist_work", "duration_home", "duration_work"]],
        ).add_to(fg)

linear.add_to(m)
plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m)
folium.LayerControl().add_to(m)

m.save("html/homes_90011.html")
m

![title](img/dist_by_zip_90011.png)